In [1]:
import pandas as pd 
import os

In [ ]:

def load_csv(file_name):
    data_path = 'C:/Users/DELL/Projects/ecommerce-sales-pipeline/data/raw'
    raw=os.path.join(data_path,file_name)
    return pd.read_csv(raw,encoding='latin1', sep=";")

Location= load_csv('Location.csv')
Products = load_csv('Products.csv')
Orders= load_csv('Orders.csv')

#reusable csv loader to avoid repeating file-loading logic across datasets
#encoding is set to latin1 because the csv is not encoded in UTF-8, hence errors when reading the csv file 
# use sep=';' so that  we can split fields. But csv are comma-delimited by default

In [3]:


def convert_type(df,desired_type):
    column = df.astype(desired_type)
    return  column

Location['Postal Code']=convert_type(Location['Postal Code'],'str')
Orders['Postal Code']=convert_type(Orders['Postal Code'],'str')

# reusable type conversion to avoid repeating multiple type conversions across datasets

In [4]:
def convert_datetype(date,dateFormat="%d/%m/%Y"):
    appropriate_dateformat = pd.to_datetime(date,format=dateFormat)

    return appropriate_dateformat


Orders['Ship Date'] = convert_datetype(Orders['Ship Date'])
Orders['Order Date'] = convert_datetype(Orders['Order Date'])


# reusable date conversion to avoid repeating multiple date conversions across datasets


In [5]:
Products[Products[['Category','Sub-Category','Product Name,,,,,']].isna().any(axis=1)]


#identifying malformed rows


,Product ID,Category,Sub-Category,"Product Name,,,,,"
26,"FUR-BO-10002916;Furniture;Bookcases;""Rush Hier...",NaN,NaN,NaN
142,"FUR-FU-10000087;Furniture;Furnishings;""Executi...",NaN,NaN,NaN
147,"FUR-FU-10000222;Furniture;Furnishings;""Seth Th...",NaN,NaN,NaN
149,"FUR-FU-10000260;Furniture;Furnishings;""6"""" Cub...",NaN,NaN,NaN
152,"FUR-FU-10000305;Furniture;Furnishings;""Tenex V...",NaN,NaN,NaN
...,...,...,...,...
1479,"OFF-SU-10004782;Office Supplies;Supplies;""Elit...",NaN,NaN,NaN
1615,"TEC-AC-10004659;Technology;Accessories;""Imatio...",NaN,NaN,NaN
1657,"TEC-MA-10001127;Technology;Machines;""HP Design...",NaN,NaN,NaN
1731,"TEC-PH-10000702;Technology;Phones;""Square Cred...",NaN,NaN,NaN


In [6]:

malformed_rows= (
    Products['Category'].isna() & 
    Products['Sub-Category'].isna() &
    Products['Product Name,,,,,'].isna())


Products.loc[malformed_rows,['Product ID', 'Category', 'Sub-Category', 'Product Name,,,,,']] = Products.loc[malformed_rows,'Product ID'].str.split(';', n=3, expand=True).values
#splitting the product is field using ; and putting the pieces into correct columns

In [7]:
Products['Product Name,,,,,'] = Products['Product Name,,,,,'].str.rstrip(',')
Products= Products.rename(columns={'Product Name,,,,,':'Product Name'})

#removing trailing commas from product name and renaming product name column

In [8]:
def profile_col(df):
    profile = {
        "Data types": df.dtypes,
        "Missing values":df.isna().sum(),
        "Missing percentage": (df.isna().mean() * 100 ).round(2),
        "Unique values": df.nunique(),
        "duplicate":df.duplicated().sum()
    }
    return pd.DataFrame(profile)

profile_col(Orders)


,Data types,Missing values,Missing percentage,Unique values,duplicate
Row ID,int64,0,0.0,9994,0
Order ID,object,0,0.0,5009,0
Order Date,datetime64[ns],0,0.0,1236,0
Ship Date,datetime64[ns],0,0.0,1334,0
Ship Mode,object,0,0.0,4,0
Customer ID,object,0,0.0,793,0
Segment,object,0,0.0,3,0
Postal Code,object,0,0.0,630,0
Product ID,object,0,0.0,1862,0
Sales,float64,0,0.0,5825,0


In [9]:
invalid_products= Orders[~Orders['Product ID'].isin(Products['Product ID'])]
print(invalid_products)

##validation -phase 8

Empty DataFrame
Columns: [Row ID, Order ID, Order Date, Ship Date, Ship Mode, Customer ID, Segment, Postal Code, Product ID, Sales, Quantity, Discount, Profit]
Index: []
